# Seq2Seq from Scratch: Neural Machine Translation

In this notebook, we'll implement a complete Sequence-to-Sequence model for translating English to French.

## What We'll Build
- **Encoder**: LSTM that reads English sentences
- **Decoder**: LSTM that generates French sentences
- **Training**: With teacher forcing
- **Inference**: With greedy decoding and beam search

Let's start!

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import numpy as np
from typing import List, Tuple
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Data Preparation

First, we need to prepare our data. We'll create a simple English-French dataset.

**Key concepts:**
- `<SOS>` (Start of Sequence): Signals the start of decoding
- `<EOS>` (End of Sequence): Signals when to stop generating
- `<PAD>`: Padding token to make sequences equal length in batches

In [ ]:
# Simple English-French dataset
# In practice, you'd use a real dataset like Multi30k or WMT
pairs = [
    ("i am cold", "j ai froid"),
    ("i am hungry", "j ai faim"),
    ("i am tired", "je suis fatigue"),
    ("i am happy", "je suis heureux"),
    ("you are cold", "tu as froid"),
    ("you are hungry", "tu as faim"),
    ("you are tired", "tu es fatigue"),
    ("he is cold", "il a froid"),
    ("he is hungry", "il a faim"),
    ("she is cold", "elle a froid"),
    ("we are happy", "nous sommes heureux"),
    ("they are tired", "ils sont fatigues"),
    ("i love you", "je t aime"),
    ("i like music", "j aime la musique"),
    ("he likes books", "il aime les livres"),
    ("she loves cats", "elle aime les chats"),
    ("we have time", "nous avons le temps"),
    ("they have money", "ils ont de l argent"),
]

print(f"Dataset size: {len(pairs)} sentence pairs")
print("\nExample pairs:")
for eng, fra in pairs[:3]:
    print(f"  English: {eng}")
    print(f"  French:  {fra}")
    print()

## 2. Building Vocabularies

We need to convert words to indices (integers) so the model can process them.

**Each vocabulary contains:**
- `word2idx`: Maps words to unique integers
- `idx2word`: Maps integers back to words
- Special tokens: `<PAD>`, `<SOS>`, `<EOS>`

In [ ]:
class Vocabulary:
    def __init__(self, name):
        self.name = name
        self.word2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2}
        self.idx2word = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>'}
        self.n_words = 3  # Count PAD, SOS, EOS
    
    def add_sentence(self, sentence):
        """Add all words in a sentence to the vocabulary"""
        for word in sentence.split():
            self.add_word(word)
    
    def add_word(self, word):
        """Add a single word to the vocabulary"""
        if word not in self.word2idx:
            self.word2idx[word] = self.n_words
            self.idx2word[self.n_words] = word
            self.n_words += 1
    
    def sentence_to_indices(self, sentence):
        """Convert sentence to list of indices"""
        return [self.word2idx[word] for word in sentence.split()]
    
    def indices_to_sentence(self, indices):
        """Convert list of indices back to sentence"""
        return ' '.join([self.idx2word[idx] for idx in indices])

# Build vocabularies
input_vocab = Vocabulary('english')
output_vocab = Vocabulary('french')

for eng, fra in pairs:
    input_vocab.add_sentence(eng)
    output_vocab.add_sentence(fra)

print(f"English vocabulary size: {input_vocab.n_words}")
print(f"French vocabulary size: {output_vocab.n_words}")
print(f"\nEnglish words: {list(input_vocab.word2idx.keys())}")
print(f"\nFrench words: {list(output_vocab.word2idx.keys())}")

## 3. The Encoder

The encoder reads the input sequence and compresses it into a context vector.

**Architecture:**
```
Input sequence → Embedding → LSTM → Final hidden state (context)
```

**Key points:**
- Embedding layer converts word indices to dense vectors
- LSTM processes the sequence step by step
- Final hidden state contains the "meaning" of the input sentence

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_size, embedding_dim, hidden_dim, num_layers=1, dropout=0.1):
        """
        Args:
            input_size: Size of input vocabulary
            embedding_dim: Dimension of word embeddings
            hidden_dim: Dimension of LSTM hidden state
            num_layers: Number of LSTM layers
            dropout: Dropout probability
        """
        super(Encoder, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Embedding layer: converts word indices to dense vectors
        # Shape: (vocab_size, embedding_dim)
        self.embedding = nn.Embedding(input_size, embedding_dim, padding_idx=0)
        
        # LSTM layer: processes the sequence
        # Input: (seq_len, batch, embedding_dim)
        # Output: (seq_len, batch, hidden_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=False  # (seq_len, batch, features) hello so this is th 
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, src):
        """
        Args:
            src: Source sequence [seq_len, batch]
        
        Returns:
            outputs: All hidden states [seq_len, batch, hidden_dim]
            hidden: Final hidden state [num_layers, batch, hidden_dim]
            cell: Final cell state [num_layers, batch, hidden_dim]
        """
        # src shape: [seq_len, batch]
        
        # Embed the input sequence
        # embedded shape: [seq_len, batch, embedding_dim]
        embedded = self.dropout(self.embedding(src))
        
        # Pass through LSTM
        # outputs: [seq_len, batch, hidden_dim] - all hidden states
        # hidden: [num_layers, batch, hidden_dim] - final hidden state
        # cell: [num_layers, batch, hidden_dim] - final cell state
        outputs, (hidden, cell) = self.lstm(embedded)
        
        # The context vector is the final hidden state and cell state
        return outputs, hidden, cell

# Test the encoder
encoder = Encoder(
    input_size=input_vocab.n_words,
    embedding_dim=256,
    hidden_dim=512,
    num_layers=1,
    dropout=0.1
).to(device)

print(encoder)
print(f"\nTotal parameters: {sum(p.numel() for p in encoder.parameters()):,}")

## 4. The Decoder

The decoder generates the output sequence one word at a time.

**Architecture:**
```
Previous word → Embedding → LSTM (initialized with encoder context) → Linear → Softmax → Next word
```

**Key points:**
- Initialized with encoder's final hidden state (context vector)
- At each step, takes previous word as input
- Outputs probability distribution over vocabulary
- During training: uses teacher forcing (feeds true previous word)
- During inference: uses its own predictions

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_size, embedding_dim, hidden_dim, num_layers=1, dropout=0.1):
        """
        Args:
            output_size: Size of output vocabulary
            embedding_dim: Dimension of word embeddings
            hidden_dim: Dimension of LSTM hidden state (must match encoder)
            num_layers: Number of LSTM layers (must match encoder)
            dropout: Dropout probability
        """
        super(Decoder, self).__init__()
        
        self.output_size = output_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Embedding layer for target words
        self.embedding = nn.Embedding(output_size, embedding_dim, padding_idx=0)
        
        # LSTM layer
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=False
        )
        
        # Output layer: maps hidden state to vocabulary probabilities
        # Shape: (hidden_dim, output_size)
        self.fc_out = nn.Linear(hidden_dim, output_size)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, input, hidden, cell):
        """
        Decode one step at a time.
        
        Args:
            input: Current input word [batch]
            hidden: Hidden state from previous step [num_layers, batch, hidden_dim]
            cell: Cell state from previous step [num_layers, batch, hidden_dim]
        
        Returns:
            prediction: Probability distribution over vocabulary [batch, output_size]
            hidden: Updated hidden state [num_layers, batch, hidden_dim]
            cell: Updated cell state [num_layers, batch, hidden_dim]
        """
        # input shape: [batch]
        # We need to add a sequence dimension
        input = input.unsqueeze(0)  # [1, batch]
        
        # Embed the input word
        # embedded shape: [1, batch, embedding_dim]
        embedded = self.dropout(self.embedding(input))
        
        # Pass through LSTM
        # output shape: [1, batch, hidden_dim]
        # hidden shape: [num_layers, batch, hidden_dim]
        # cell shape: [num_layers, batch, hidden_dim]
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        
        # Remove sequence dimension from output
        # output shape: [batch, hidden_dim]
        output = output.squeeze(0)
        
        # Generate prediction
        # prediction shape: [batch, output_size]
        prediction = self.fc_out(output)
        
        return prediction, hidden, cell

# Test the decoder
decoder = Decoder(
    output_size=output_vocab.n_words,
    embedding_dim=256,
    hidden_dim=512,
    num_layers=1,
    dropout=0.1
).to(device)

print(decoder)
print(f"\nTotal parameters: {sum(p.numel() for p in decoder.parameters()):,}")

## 5. The Complete Seq2Seq Model

Now we combine the encoder and decoder into a single model.

**Training flow:**
1. Encoder processes input sequence → produces context vector
2. Decoder initialized with context vector
3. For each target position:
   - Decoder takes previous word (teacher forcing)
   - Produces next word prediction
   - Compute loss against true word

**Teacher forcing ratio:**
- During training, we sometimes use decoder's own predictions instead of true words
- This helps the model learn to recover from mistakes
- Typical value: 0.5 (50% of the time use true words)

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
        # Make sure encoder and decoder have same hidden dimensions
        assert encoder.hidden_dim == decoder.hidden_dim, \
            "Encoder and decoder must have same hidden dimensions!"
        assert encoder.num_layers == decoder.num_layers, \
            "Encoder and decoder must have same number of layers!"
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        """
        Args:
            src: Source sequence [src_len, batch]
            trg: Target sequence [trg_len, batch]
            teacher_forcing_ratio: Probability of using teacher forcing
        
        Returns:
            outputs: Predictions for each position [trg_len, batch, output_size]
        """
        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_size
        
        # Tensor to store decoder outputs
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        
        # Encode the source sequence
        # enc_outputs: [src_len, batch, hidden_dim] - not used in basic seq2seq
        # hidden: [num_layers, batch, hidden_dim] - context vector
        # cell: [num_layers, batch, hidden_dim] - context vector
        enc_outputs, hidden, cell = self.encoder(src)
        
        # First input to decoder is <SOS> token
        input = trg[0, :]  # [batch]
        
        # Decode one step at a time
        for t in range(1, trg_len):
            # Decode current step
            # output: [batch, output_size]
            # hidden: [num_layers, batch, hidden_dim]
            # cell: [num_layers, batch, hidden_dim]
            output, hidden, cell = self.decoder(input, hidden, cell)
            
            # Store prediction
            outputs[t] = output
            
            # Get the highest predicted token
            top1 = output.argmax(1)  # [batch]
            
            # Teacher forcing: use true target as next input
            # Or use own prediction
            teacher_force = random.random() < teacher_forcing_ratio
            input = trg[t] if teacher_force else top1
        
        return outputs

# Create the complete model
model = Seq2Seq(encoder, decoder, device).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Data Preparation Functions

We need functions to convert our sentence pairs into tensors that PyTorch can process.

In [ ]:
def indices_from_sentence(vocab, sentence):
    """Convert sentence to list of indices"""
    return [vocab.word2idx[word] for word in sentence.split()]

def tensor_from_sentence(vocab, sentence):
    """Convert sentence to tensor of indices, with EOS token"""
    indices = indices_from_sentence(vocab, sentence)
    indices.append(vocab.word2idx['<EOS>'])
    return torch.tensor(indices, dtype=torch.long, device=device).view(-1, 1)

def tensors_from_pair(pair):
    """Convert a pair of sentences to tensors"""
    input_tensor = tensor_from_sentence(input_vocab, pair[0])
    target_tensor = tensor_from_sentence(output_vocab, pair[1])
    return (input_tensor, target_tensor)

# Test
test_pair = pairs[0]
input_tensor, target_tensor = tensors_from_pair(test_pair)
print(f"English: {test_pair[0]}")
print(f"Input tensor shape: {input_tensor.shape}")
print(f"Input tensor: {input_tensor.squeeze().tolist()}")
print(f"\nFrench: {test_pair[1]}")
print(f"Target tensor shape: {target_tensor.shape}")
print(f"Target tensor: {target_tensor.squeeze().tolist()}")
print(f"\nDecoded back:")
print(f"English: {input_vocab.indices_to_sentence(input_tensor.squeeze().tolist()[:-1])}")
print(f"French: {output_vocab.indices_to_sentence(target_tensor.squeeze().tolist()[:-1])}")

## 7. Training Function

**Training process:**
1. Forward pass through encoder and decoder
2. Compute loss (cross-entropy between predictions and targets)
3. Backward pass (compute gradients)
4. Update weights

**Loss calculation:**
- We ignore padding tokens in the loss
- We skip the first position (which is `<SOS>`)
- Cross-entropy measures how well predictions match true words

In [ ]:
def train_step(model, iterator, optimizer, criterion, clip):
    """
    Train the model for one step.
    
    Args:
        model: Seq2Seq model
        iterator: Data iterator
        optimizer: Optimizer
        criterion: Loss function
        clip: Gradient clipping value
    
    Returns:
        loss: Average loss for this step
    """
    model.train()
    
    epoch_loss = 0
    
    for src, trg in iterator:
        # src: [src_len, batch]
        # trg: [trg_len, batch]
        
        optimizer.zero_grad()
        
        # Forward pass
        # output: [trg_len, batch, output_size]
        output = model(src, trg)
        
        # Reshape for loss calculation
        # We skip the first token (<SOS>) in the target
        output_dim = output.shape[-1]
        
        # Flatten: [trg_len * batch, output_size]
        output = output[1:].view(-1, output_dim)
        # Flatten: [trg_len * batch]
        trg = trg[1:].view(-1)
        
        # Calculate loss
        # Cross-entropy between predictions and true targets
        loss = criterion(output, trg)
        
        # Backward pass
        loss.backward()
        
        # Clip gradients to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        
        # Update weights
        optimizer.step()
        
        epoch_loss += loss.item()
    
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    """
    Evaluate the model (no gradient updates).
    """
    model.eval()
    
    epoch_loss = 0
    
    with torch.no_grad():
        for src, trg in iterator:
            # No teacher forcing during evaluation
            output = model(src, trg, teacher_forcing_ratio=0)
            
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)
            
            loss = criterion(output, trg)
            
            epoch_loss += loss.item()
    
    return epoch_loss / len(iterator)

print("Training functions defined!")

## 8. Training the Model

Now let's train! Since our dataset is small, training will be fast.

**Hyperparameters:**
- Learning rate: 0.001
- Gradient clipping: 1.0 (prevents exploding gradients)
- Loss: Cross-entropy (ignoring padding)
- Optimizer: Adam

In [ ]:
# Prepare training data
training_pairs = [tensors_from_pair(pair) for pair in pairs]

# Simple data loader (since dataset is small)
def get_batch_iterator(pairs, batch_size=4):
    """Create batches from pairs"""
    random.shuffle(pairs)
    batches = []
    
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i+batch_size]
        
        # Find max lengths in this batch
        src_max = max([pair[0].shape[0] for pair in batch])
        trg_max = max([pair[1].shape[0] for pair in batch])
        
        # Pad sequences
        src_batch = []
        trg_batch = []
        
        for src, trg in batch:
            # Pad source
            src_padded = F.pad(src, (0, 0, 0, src_max - src.shape[0]), value=0)
            src_batch.append(src_padded)
            
            # Pad target
            trg_padded = F.pad(trg, (0, 0, 0, trg_max - trg.shape[0]), value=0)
            trg_batch.append(trg_padded)
        
        # Stack into tensors [seq_len, batch]
        src_batch = torch.cat(src_batch, dim=1)
        trg_batch = torch.cat(trg_batch, dim=1)
        
        batches.append((src_batch, trg_batch))
    
    return batches

# Initialize optimizer and loss
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding

# Training loop
N_EPOCHS = 100
CLIP = 1
BATCH_SIZE = 4

train_losses = []
print("Starting training...\n")

for epoch in range(N_EPOCHS):
    iterator = get_batch_iterator(training_pairs, BATCH_SIZE)
    
    train_loss = train_step(model, iterator, optimizer, criterion, CLIP)
    train_losses.append(train_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch: {epoch+1:03} | Train Loss: {train_loss:.3f}')

print("\nTraining complete!")

## 9. Visualize Training

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses)
plt.title('Training Loss Over Time')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

## 10. Inference: Greedy Decoding

During inference, we don't have the target sequence. The decoder must generate words using only its own predictions.

**Greedy decoding:**
- At each step, pick the word with highest probability
- Use that word as input for next step
- Stop when `<EOS>` is generated or max length reached

In [ ]:
def translate_sentence(model, sentence, max_length=20):
    """
    Translate a sentence using greedy decoding.
    
    Args:
        model: Trained Seq2Seq model
        sentence: Input sentence (string)
        max_length: Maximum output length
    
    Returns:
        translated_sentence: Output sentence (string)
        attention_weights: None (for compatibility with attention models)
    """
    model.eval()
    
    with torch.no_grad():
        # Prepare input
        src_tensor = tensor_from_sentence(input_vocab, sentence)
        # src_tensor: [src_len, 1]
        
        # Encode
        enc_outputs, hidden, cell = model.encoder(src_tensor)
        
        # Start with <SOS> token
        decoder_input = torch.tensor([output_vocab.word2idx['<SOS>']], device=device)
        
        decoded_words = []
        
        for _ in range(max_length):
            # Decode one step
            output, hidden, cell = model.decoder(decoder_input, hidden, cell)
            
            # Get the word with highest probability
            top1 = output.argmax(1).item()
            
            # Check if we generated <EOS>
            if top1 == output_vocab.word2idx['<EOS>']:
                break
            
            decoded_words.append(output_vocab.idx2word[top1])
            
            # Use this word as next input
            decoder_input = torch.tensor([top1], device=device)
        
        return ' '.join(decoded_words), None

# Test on training data
print("Testing on training examples:\n")
for eng, fra in pairs[:5]:
    translation, _ = translate_sentence(model, eng)
    print(f"English: {eng}")
    print(f"True French: {fra}")
    print(f"Predicted: {translation}")
    print()

## 11. Inference: Beam Search

Greedy decoding has a problem: it might pick a word that looks good now but leads to poor translations later.

**Beam search** keeps track of multiple possible translations at once:
1. Start with `<SOS>`
2. At each step, keep the top k most probable sequences (beam width = k)
3. Expand each sequence by all possible next words
4. Keep only the top k expanded sequences
5. Stop when all beams generate `<EOS>`

**Example with beam_width=3:**
```
Step 1: ["je", "tu", "il"] (top 3 words)
Step 2: ["je suis", "je t", "tu es"] (top 3 sequences)
Step 3: ["je suis heureux", "je t aime", "je suis fatigue"]
```

In [ ]:
def beam_search_decode(model, sentence, beam_width=3, max_length=20):
    """
    Translate a sentence using beam search.
    
    Args:
        model: Trained Seq2Seq model
        sentence: Input sentence (string)
        beam_width: Number of beams to keep
        max_length: Maximum output length
    
    Returns:
        best_sequence: Best translation (string)
        best_score: Score of best translation
    """
    model.eval()
    
    with torch.no_grad():
        # Encode input
        src_tensor = tensor_from_sentence(input_vocab, sentence)
        enc_outputs, hidden, cell = model.encoder(src_tensor)
        
        # Initialize beams
        # Each beam: (sequence, score, hidden, cell)
        sos_idx = output_vocab.word2idx['<SOS>']
        eos_idx = output_vocab.word2idx['<EOS>']
        
        beams = [([sos_idx], 0.0, hidden, cell)]
        completed = []
        
        for _ in range(max_length):
            candidates = []
            
            for sequence, score, hidden, cell in beams:
                # If this beam already ended, keep it as is
                if sequence[-1] == eos_idx:
                    completed.append((sequence, score))
                    continue
                
                # Get last word
                last_word = torch.tensor([sequence[-1]], device=device)
                
                # Decode one step
                output, new_hidden, new_cell = model.decoder(last_word, hidden, cell)
                
                # Get log probabilities
                log_probs = F.log_softmax(output, dim=1)
                
                # Get top k candidates
                top_log_probs, top_indices = log_probs.topk(beam_width)
                
                # Expand beam
                for log_prob, idx in zip(top_log_probs[0], top_indices[0]):
                    new_sequence = sequence + [idx.item()]
                    new_score = score + log_prob.item()
                    candidates.append((new_sequence, new_score, new_hidden, new_cell))
            
            # Keep top beams
            if not candidates:
                break
            
            beams = sorted(candidates, key=lambda x: x[1] / len(x[0]), reverse=True)[:beam_width]
            
            # Check if all beams are complete
            if all(b[0][-1] == eos_idx for b in beams):
                completed.extend([(b[0], b[1]) for b in beams])
                break
        
        # Add remaining beams to completed
        completed.extend([(b[0], b[1]) for b in beams])
        
        # Get best sequence (normalize by length)
        best_sequence, best_score = max(completed, key=lambda x: x[1] / len(x[0]))
        
        # Convert to words
        decoded_words = []
        for idx in best_sequence[1:]:  # Skip <SOS>
            if idx == eos_idx:
                break
            decoded_words.append(output_vocab.idx2word[idx])
        
        return ' '.join(decoded_words), best_score

# Test beam search
print("Testing beam search:\n")
for eng, fra in pairs[:5]:
    greedy_translation, _ = translate_sentence(model, eng)
    beam_translation, beam_score = beam_search_decode(model, eng, beam_width=3)
    
    print(f"English: {eng}")
    print(f"True French: {fra}")
    print(f"Greedy: {greedy_translation}")
    print(f"Beam (width=3): {beam_translation} (score: {beam_score:.2f})")
    print()

## 12. Analysis: The Bottleneck Problem

Let's visualize why basic seq2seq struggles with long sequences.

The entire input sentence must be compressed into a single fixed-size vector (the hidden state). As sentences get longer, information gets lost.

In [ ]:
def analyze_context_vector(model, sentence):
    """
    Visualize the context vector produced by the encoder.
    """
    model.eval()
    
    with torch.no_grad():
        src_tensor = tensor_from_sentence(input_vocab, sentence)
        enc_outputs, hidden, cell = model.encoder(src_tensor)
        
        # Get context vector (final hidden state)
        context = hidden[-1].squeeze().cpu().numpy()
        
        print(f"Sentence: {sentence}")
        print(f"Sentence length: {len(sentence.split())} words")
        print(f"Context vector shape: {context.shape}")
        print(f"Context vector norm: {np.linalg.norm(context):.2f}")
        
        # Visualize
        plt.figure(figsize=(12, 2))
        plt.imshow(context.reshape(1, -1), cmap='coolwarm', aspect='auto')
        plt.title(f'Context Vector for: "{sentence}"')
        plt.xlabel('Hidden Dimension')
        plt.colorbar()
        plt.show()

# Analyze different length sentences
analyze_context_vector(model, "i am cold")
analyze_context_vector(model, "i love you")
analyze_context_vector(model, "they have money")

## 13. Summary and Next Steps

**What we built:**
- ✅ Encoder-Decoder architecture
- ✅ Training with teacher forcing
- ✅ Greedy decoding
- ✅ Beam search

**Key limitations of basic seq2seq:**
1. **Fixed-length context vector**: All input information must fit in one vector
2. **Information loss**: Long sequences lose information from early words
3. **No alignment**: Decoder can't "look back" at specific input words

**This is exactly why attention was invented!**

**Bahdanau Attention (coming next) solves this by:**
- Letting the decoder access *all* encoder hidden states, not just the last one
- Learning to "attend" to relevant input words at each decoding step
- Creating a weighted context vector for each output position

---

## Exercises to Deepen Understanding

1. **Modify teacher forcing ratio**: Try training with different values (0.0, 0.5, 1.0)
2. **Experiment with hidden dimensions**: Try 256, 512, 1024
3. **Add more layers**: Change `num_layers` to 2 or 3
4. **Try different beam widths**: Compare beam_width=1, 3, 5, 10
5. **Visualize attention**: (Wait for Bahdanau notebook!)

Ready to move on to Bahdanau attention?

In [ ]:
# Save the model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'input_vocab': input_vocab,
    'output_vocab': output_vocab,
}, '/home/claude/seq2seq_model.pth')

print("Model saved!")
print("\nYou now understand the fundamentals of seq2seq!")
print("Next up: Bahdanau Attention 🎯")